# Warstwa 4 (Bonus): Machine Learning

**Cel:** Predykcja kategorii popularności wideo (`low / medium / high / viral`)  
**Model:** Random Forest + XGBoost  
**Wejście:** `data/parquet/trending_clean.parquet`

### Cechy użyte do predykcji
- `category_id` — kategoria tematyczna
- `region` — kraj
- `tag_count` — liczba tagów
- `title_length` — długość tytułu
- `title_word_count` — liczba słów
- `title_has_caps` — czy tytuł zawiera CAPS
- `title_has_number` — czy tytuł zawiera cyfry
- `publish_hour` — godzina publikacji
- `publish_dow` — dzień tygodnia
- `publish_month` — miesiąc
- `duration_seconds` — długość wideo
- `is_hd`, `has_caption` — techniczne cechy wideo


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xgboost as xgb

warnings.filterwarnings("ignore")
PARQUET_DIR = Path("../data/parquet")

print("Biblioteki OK")

Biblioteki OK


In [2]:
# ── Dane ──────────────────────────────────────────────────────────────────────
df = pq.read_table(PARQUET_DIR / "trending_clean.parquet").to_pandas()

FEATURES = [
    "category_id", "region", "tag_count", "title_length",
    "title_word_count", "title_has_caps", "title_has_number",
    "publish_hour", "publish_dow", "publish_month",
    "duration_seconds", "is_hd", "has_caption"
]
TARGET = "view_category"

df_ml = df[FEATURES + [TARGET]].dropna()

# Encode kategoryczne
le_region   = LabelEncoder()
le_cat_id   = LabelEncoder()
le_target   = LabelEncoder()

df_ml = df_ml.copy()
df_ml["region"]      = le_region.fit_transform(df_ml["region"].astype(str))
df_ml["category_id"] = le_cat_id.fit_transform(df_ml["category_id"].astype(str))

X = df_ml[FEATURES]
y = le_target.fit_transform(df_ml[TARGET])

print(f"Dane: {X.shape[0]} wierszy, {X.shape[1]} cech")
print(f"Klasy: {list(le_target.classes_)}")
print(f"Rozkład klas:")
for cls, cnt in zip(le_target.classes_, np.bincount(y)):
    print(f"  {cls}: {cnt} ({100*cnt/len(y):.1f}%)")

Dane: 874 wierszy, 13 cech
Klasy: ['high', 'low', 'medium', 'viral']
Rozkład klas:
  high: 34 (3.9%)
  low: 505 (57.8%)
  medium: 330 (37.8%)
  viral: 5 (0.6%)


In [3]:
# ── Podział na zbiory ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

Train: 699  Test: 175


In [4]:
# ── Model 1: Random Forest ────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=12,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf    = accuracy_score(y_test, y_pred_rf)

cv_rf = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print(f"Random Forest:")
print(f"  Dokładność (test):   {acc_rf:.4f}")
print(f"  Dokładność (5-fold): {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))

Random Forest:
  Dokładność (test):   0.6971
  Dokładność (5-fold): 0.6786 ± 0.0617

              precision    recall  f1-score   support

        high       1.00      0.29      0.44         7
         low       0.74      0.83      0.78       101
      medium       0.61      0.55      0.58        66
       viral       0.00      0.00      0.00         1

    accuracy                           0.70       175
   macro avg       0.59      0.42      0.45       175
weighted avg       0.70      0.70      0.69       175



In [5]:
# ── Model 2: XGBoost ──────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric="mlogloss",
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
acc_xgb    = accuracy_score(y_test, y_pred_xgb)

cv_xgb = cross_val_score(xgb_model, X, y, cv=5, scoring="accuracy")
print(f"XGBoost:")
print(f"  Dokładność (test):   {acc_xgb:.4f}")
print(f"  Dokładność (5-fold): {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")

XGBoost:
  Dokładność (test):   0.6629
  Dokładność (5-fold): 0.6339 ± 0.0744


In [6]:
# ── Ważność cech (najlepszy model) ────────────────────────────────────────────
best_model    = rf if acc_rf >= acc_xgb else xgb_model
best_name     = "Random Forest" if acc_rf >= acc_xgb else "XGBoost"
importances   = pd.Series(best_model.feature_importances_, index=FEATURES)
importances   = importances.sort_values(ascending=True)

fig = px.bar(
    x=importances.values, y=importances.index,
    orientation="h",
    title=f"Ważność cech — {best_name}",
    labels={"x": "Ważność", "y": "Cecha"},
    color=importances.values,
    color_continuous_scale="Blues"
)
fig.update_layout(showlegend=False, height=450)
fig.show()

In [7]:
# ── Macierz pomyłek ───────────────────────────────────────────────────────────
cm   = confusion_matrix(y_test, rf.predict(X_test))
lbls = le_target.classes_

fig = px.imshow(
    cm,
    x=lbls, y=lbls,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Macierz pomyłek — Random Forest",
    labels={"x": "Przewidywana", "y": "Rzeczywista"}
)
fig.show()

In [8]:
# ── Porównanie modeli ─────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Model":    ["Random Forest", "XGBoost"],
    "Test Acc": [round(acc_rf, 4),  round(acc_xgb, 4)],
    "CV Mean":  [round(cv_rf.mean(), 4),  round(cv_xgb.mean(), 4)],
    "CV Std":   [round(cv_rf.std(), 4),   round(cv_xgb.std(), 4)],
})

fig = px.bar(
    comparison, x="Model", y="Test Acc",
    error_y="CV Std",
    title="Porównanie dokładności modeli",
    color="Model",
    text="Test Acc",
    range_y=[0, 1]
)
fig.update_traces(textposition="outside")
fig.show()

print(f"\nLepszy model: {best_name} (acc={max(acc_rf, acc_xgb):.4f})")


Lepszy model: Random Forest (acc=0.6971)
